<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Protein_Structure_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Protein Structure Prediction

# Project 5 Bioinformatics

Is notebook mein hum protein amino acid sequence se uski secondary structure class predict karain gay Alpha-helix dominant, Beta-sheet dominant, ya Mixed/Coil amino acid composition, physicochemical properties, aur Chou-Fasman-style propensities use kar ke.

# **Pipeline:**
# 1. Protein sequence dataset generate karna (biologically-grounded amino acid propensities se)
# 2. Feature extraction — amino acid composition, hydrophobicity, charge, molecular weight
# 3. Interactive Plotly visualizations
# 4. Model training (multi-class classification)
# 5. Evaluation (accuracy, confusion matrix, feature importance)
# 6. Runtime cell apni khud ki protein sequence daal kar direct predict karein + secondary structure content estimate (%Helix/%Sheet/%Coil)




## 1. Setup & Imports

In [1]:
# !pip install -q plotly scikit-learn pandas numpy ipywidgets

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Reference Biochemical Data



In [2]:
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

# Kyte-Doolittle hydrophobicity
hydrophobicity = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8,
    'P': -1.6, 'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

# Approximate molecular weight (Da)
mol_weight = {
    'A': 89.1, 'R': 174.2, 'N': 132.1, 'D': 133.1, 'C': 121.2, 'Q': 146.2, 'E': 147.1,
    'G': 75.1, 'H': 155.2, 'I': 131.2, 'L': 131.2, 'K': 146.2, 'M': 149.2, 'F': 165.2,
    'P': 115.1, 'S': 105.1, 'T': 119.1, 'W': 204.2, 'Y': 181.2, 'V': 117.1
}

# Net charge at physiological pH (simplified)
charge = {'D': -1, 'E': -1, 'K': 1, 'R': 1, 'H': 0.1}
charge = {aa: charge.get(aa, 0) for aa in amino_acids}

# Chou-Fasman propensities (Pa=helix, Pb=sheet, Pc=coil/turn) — classic reference values
chou_fasman = {
    'A': {'Pa': 1.42, 'Pb': 0.83, 'Pc': 0.66}, 'R': {'Pa': 0.98, 'Pb': 0.93, 'Pc': 0.95},
    'N': {'Pa': 0.67, 'Pb': 0.89, 'Pc': 1.56}, 'D': {'Pa': 1.01, 'Pb': 0.54, 'Pc': 1.46},
    'C': {'Pa': 0.70, 'Pb': 1.19, 'Pc': 1.19}, 'Q': {'Pa': 1.11, 'Pb': 1.10, 'Pc': 0.98},
    'E': {'Pa': 1.51, 'Pb': 0.37, 'Pc': 0.74}, 'G': {'Pa': 0.57, 'Pb': 0.75, 'Pc': 1.56},
    'H': {'Pa': 1.00, 'Pb': 0.87, 'Pc': 0.95}, 'I': {'Pa': 1.08, 'Pb': 1.60, 'Pc': 0.47},
    'L': {'Pa': 1.21, 'Pb': 1.30, 'Pc': 0.59}, 'K': {'Pa': 1.16, 'Pb': 0.74, 'Pc': 1.01},
    'M': {'Pa': 1.45, 'Pb': 1.05, 'Pc': 0.60}, 'F': {'Pa': 1.13, 'Pb': 1.38, 'Pc': 0.60},
    'P': {'Pa': 0.57, 'Pb': 0.55, 'Pc': 1.52}, 'S': {'Pa': 0.77, 'Pb': 0.75, 'Pc': 1.43},
    'T': {'Pa': 0.83, 'Pb': 1.19, 'Pc': 0.96}, 'W': {'Pa': 1.08, 'Pb': 1.37, 'Pc': 0.96},
    'Y': {'Pa': 0.69, 'Pb': 1.47, 'Pc': 1.14}, 'V': {'Pa': 1.06, 'Pb': 1.70, 'Pc': 0.50},
}

print("Reference tables loaded: hydrophobicity, molecular weight, charge, Chou-Fasman propensities")


Reference tables loaded: hydrophobicity, molecular weight, charge, Chou-Fasman propensities


## 3. Generate Protein Sequence Dataset

In [3]:
helix_favoring = [aa for aa in amino_acids if chou_fasman[aa]['Pa'] > 1.1]
sheet_favoring = [aa for aa in amino_acids if chou_fasman[aa]['Pb'] > 1.2]
coil_favoring = [aa for aa in amino_acids if chou_fasman[aa]['Pc'] > 1.1]

def generate_protein(seq_len, favored_aas, favor_strength, rng):
    seq = []
    for _ in range(seq_len):
        if rng.random() < favor_strength:
            seq.append(rng.choice(favored_aas))
        else:
            seq.append(rng.choice(amino_acids))
    return "".join(seq)

def generate_dataset(n_per_class=150, min_len=80, max_len=200, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    classes = {
        "Alpha-helix dominant": helix_favoring,
        "Beta-sheet dominant": sheet_favoring,
        "Mixed/Coil": coil_favoring,
    }
    for cls, favored in classes.items():
        for _ in range(n_per_class):
            length = rng.integers(min_len, max_len)
            seq = generate_protein(length, favored, favor_strength=0.45, rng=rng)
            rows.append({"sequence": seq, "structure_class": cls})
    return pd.DataFrame(rows)

protein_df = generate_dataset(n_per_class=150)
print(f"Total proteins: {len(protein_df)}")
protein_df.head()


Total proteins: 450


,sequence,structure_class
0,MCLRSMKSPKFQTRIQMLAWRQIKDKLPLKMELMALRMKKQCSCKF...,Alpha-helix dominant
1,KLELENQHEAMDLLRRLQAKLMPGQQFKCRDKHLMMWEVERQQSKE...,Alpha-helix dominant
2,LHFNEKAEEMLQMMAEFEAVDFQTWNSRDYVLLKKFFKIMWSHLQS...,Alpha-helix dominant
3,QEDQKQEMPEFPFKKINNLHKMINVRMKTACAEMHKTLEKIKMDLH...,Alpha-helix dominant
4,FLKKMKIEHMPLFFAGNMWYQWEKHYAAFAQLAAMCQFEAEAAFLM...,Alpha-helix dominant


## 4. Feature Extraction

In [4]:
def extract_protein_features(seq):
    length = len(seq)
    composition = {f"frac_{aa}": seq.count(aa) / length for aa in amino_acids}
    avg_hydro = np.mean([hydrophobicity[aa] for aa in seq])
    avg_mw = np.mean([mol_weight[aa] for aa in seq])
    net_charge = sum(charge[aa] for aa in seq)
    avg_pa = np.mean([chou_fasman[aa]['Pa'] for aa in seq])
    avg_pb = np.mean([chou_fasman[aa]['Pb'] for aa in seq])
    avg_pc = np.mean([chou_fasman[aa]['Pc'] for aa in seq])

    features = {**composition,
                "avg_hydrophobicity": avg_hydro, "avg_mol_weight": avg_mw,
                "net_charge": net_charge, "length": length,
                "avg_helix_propensity": avg_pa, "avg_sheet_propensity": avg_pb,
                "avg_coil_propensity": avg_pc}
    return features

feature_rows = protein_df["sequence"].apply(extract_protein_features)
protein_features = pd.DataFrame(list(feature_rows))
protein_features["structure_class"] = protein_df["structure_class"]

print(f"Feature matrix shape: {protein_features.shape}")
protein_features.head()


Feature matrix shape: (450, 28)


,frac_A,frac_C,frac_D,frac_E,frac_F,frac_G,frac_H,frac_I,frac_K,frac_L,...,frac_W,frac_Y,avg_hydrophobicity,avg_mol_weight,net_charge,length,avg_helix_propensity,avg_sheet_propensity,avg_coil_propensity,structure_class
0,0.044444,0.044444,0.033333,0.088889,0.088889,0.033333,0.000000,0.022222,0.133333,0.111111,...,0.022222,0.022222,-0.443333,140.514444,6.0,90,1.124556,0.971444,0.901111,Alpha-helix dominant
1,0.073171,0.024390,0.048780,0.121951,0.060976,0.024390,0.024390,0.000000,0.097561,0.146341,...,0.024390,0.024390,-0.536585,139.797561,-1.8,82,1.158659,0.968902,0.872561,Alpha-helix dominant
2,0.102190,0.029197,0.065693,0.065693,0.072993,0.014599,0.029197,0.014599,0.065693,0.102190,...,0.043796,0.029197,-0.127007,138.232847,-4.6,137,1.130657,1.004964,0.888102,Alpha-helix dominant
3,0.104167,0.020833,0.013889,0.076389,0.076389,0.027778,0.048611,0.041667,0.097222,0.083333,...,0.027778,0.034722,-0.247222,137.001389,4.7,144,1.110208,1.011528,0.880417,Alpha-helix dominant
4,0.125000,0.019231,0.019231,0.096154,0.096154,0.019231,0.038462,0.019231,0.076923,0.163462,...,0.019231,0.048077,0.212500,137.686538,-3.6,104,1.176923,1.014615,0.821442,Alpha-helix dominant


## 5. Interactive Visualizations

In [5]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("Helix Propensity", "Sheet Propensity", "Coil Propensity"))
colors = {"Alpha-helix dominant": "#E63946", "Beta-sheet dominant": "#2E86AB", "Mixed/Coil": "#43AA8B"}

for i, col in enumerate(["avg_helix_propensity", "avg_sheet_propensity", "avg_coil_propensity"]):
    for cls in colors:
        vals = protein_features.loc[protein_features["structure_class"] == cls, col]
        fig.add_trace(go.Box(y=vals, name=cls, marker_color=colors[cls], showlegend=(i == 0)), row=1, col=i+1)

fig.update_layout(height=500, title_text="Structural Propensities by Class")
fig.show()


In [6]:
fig = px.scatter(
    protein_features, x="avg_hydrophobicity", y="net_charge", color="structure_class",
    size="avg_mol_weight", hover_data=["length"],
    title="Hydrophobicity vs Net Charge (bubble size = avg. molecular weight)",
    template="plotly_white", color_discrete_map=colors
)
fig.update_layout(height=550)
fig.show()

# PCA on amino acid composition
comp_cols = [f"frac_{aa}" for aa in amino_acids]
pca = PCA(n_components=2)
pcs = pca.fit_transform(StandardScaler().fit_transform(protein_features[comp_cols]))
pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
pca_df["structure_class"] = protein_features["structure_class"]

fig2 = px.scatter(
    pca_df, x="PC1", y="PC2", color="structure_class",
    title=f"PCA of Amino Acid Composition (PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)",
    template="plotly_white", color_discrete_map=colors
)
fig2.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))
fig2.update_layout(height=550)
fig2.show()


In [7]:
avg_comp = protein_features.groupby("structure_class")[comp_cols].mean()
avg_comp.columns = amino_acids

fig = px.imshow(
    avg_comp.values, x=amino_acids, y=avg_comp.index,
    color_continuous_scale="Viridis", aspect="auto", text_auto=".2f",
    title="Average Amino Acid Composition by Structural Class",
    labels=dict(color="Fraction")
)
fig.update_layout(height=400)
fig.show()


## 6. Train Classifier

In [8]:
le = LabelEncoder()
X = protein_features.drop(columns=["structure_class"])
y = le.fit_transform(protein_features["structure_class"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

protein_scaler = StandardScaler()
X_train_scaled = protein_scaler.fit_transform(X_train)
X_test_scaled = protein_scaler.transform(X_test)

protein_clf = RandomForestClassifier(n_estimators=300, random_state=42)
protein_clf.fit(X_train_scaled, y_train)

preds = protein_clf.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}\n")
print(classification_report(y_test, preds, target_names=le.classes_))


Accuracy: 1.000

                      precision    recall  f1-score   support

Alpha-helix dominant       1.00      1.00      1.00        37
 Beta-sheet dominant       1.00      1.00      1.00        38
          Mixed/Coil       1.00      1.00      1.00        38

            accuracy                           1.00       113
           macro avg       1.00      1.00      1.00       113
        weighted avg       1.00      1.00      1.00       113



In [9]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                 x=le.classes_, y=le.classes_,
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 title="Confusion Matrix — Protein Structure Classifier")
fig.update_layout(height=500, width=600)
fig.show()

feature_names = X.columns
importances = pd.Series(protein_clf.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
fig2 = px.bar(importances, orientation='h', title="Top 15 Feature Importances",
              labels={"value": "Importance", "index": "Feature"}, template="plotly_white",
              color=importances.values, color_continuous_scale="Viridis")
fig2.update_layout(height=500, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 7.  Runtime Prediction — Apni Protein Sequence Direct Input Karein


In [10]:
seq_input_box = widgets.Textarea(
    value="MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWELVMGDGE",
    description="Protein Sequence:",
    placeholder="Amino acid single-letter codes likhein (e.g. MKTAYIAK...)",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px', height='90px')
)

predict_btn = widgets.Button(description=" Structure Predict Karein", button_style='success',
                              layout=widgets.Layout(width='260px', height='42px'))
out = widgets.Output()

def render_result(label, proba, classes, pa, pb, pc):
    color_map = colors
    top_color = color_map.get(label, "#555")
    total_prop = pa + pb + pc
    pct_helix = pa / total_prop * 100
    pct_sheet = pb / total_prop * 100
    pct_coil = pc / total_prop * 100

    prob_rows = "".join(
        f'<div style="display:flex; justify-content:space-between; font-size:13px; margin-top:3px;">'
        f'<span>{cls}</span><span><b>{p*100:.2f}%</b></span></div>'
        for cls, p in zip(classes, proba)
    )

    html = f"""
    <div style="border:2px solid {top_color}; border-radius:12px; padding:18px; margin-top:10px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:21px; font-weight:700; color:{top_color};"> Predicted Class: {label}</div>
        <div style="font-size:13px; margin-top:10px; color:#333;"><b>Class Probabilities:</b></div>
        {prob_rows}
        <hr style="margin:12px 0; border:none; border-top:1px solid #ddd;">
        <div style="font-size:13px; color:#333;"><b>Chou-Fasman Structure Content Estimate:</b></div>
        <div style="display:flex; height:20px; width:100%; border-radius:6px; overflow:hidden; margin-top:6px;">
            <div style="width:{pct_helix:.1f}%; background:#E63946;" title="Helix"></div>
            <div style="width:{pct_sheet:.1f}%; background:#2E86AB;" title="Sheet"></div>
            <div style="width:{pct_coil:.1f}%; background:#43AA8B;" title="Coil"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>🔴 Helix: {pct_helix:.1f}%</span><span>🔵 Sheet: {pct_sheet:.1f}%</span><span>🟢 Coil: {pct_coil:.1f}%</span>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        raw = seq_input_box.value.strip().upper()
        seq_clean = "".join([c for c in raw if c in amino_acids])

        if len(seq_clean) < 10:
            print(" Sequence bahut chhoti hai (kam az kam 10 valid amino acids chahiye).")
            return

        feats = extract_protein_features(seq_clean)
        row_df = pd.DataFrame([feats])[X.columns]
        row_scaled = protein_scaler.transform(row_df)

        pred = protein_clf.predict(row_scaled)[0]
        proba = protein_clf.predict_proba(row_scaled)[0]
        label = le.inverse_transform([pred])[0]

        render_result(label, proba, le.classes_, feats["avg_helix_propensity"],
                      feats["avg_sheet_propensity"], feats["avg_coil_propensity"])

predict_btn.on_click(on_predict)

display(seq_input_box)
display(predict_btn)
display(out)


Textarea(value='MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYT…

Button(button_style='success', description=' Structure Predict Karein', layout=Layout(height='42px', width='26…

Output()